# Notebook 03 — Perguntas analíticas

Responde às perguntas 1–8, 10 e 11 da orientação do Datathon. A pergunta 9 (modelo preditivo) é respondida no notebook 02 — o modelo treinado *é* o instrumento analítico daquela pergunta.

**Base:** mesmo recorte do modelo (`painel_modelo.csv`, 2.566 linhas). Ficam de fora: matrículas em processamento, linhas sem features completas, movimentos atípicos de fase e a Fase 8 nas análises de índices. Os universitários entram apenas no bloco de trajetória da pergunta 10, com transição documentada ano a ano.

Cada figura é salva em `reports/figuras/` e acompanhada do insight em texto.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA = '/content/drive/MyDrive/Datathon/'
except ImportError:
    PASTA = './'

import os
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

PASTA_FIG = os.path.join(PASTA, 'reports', 'figuras')
os.makedirs(PASTA_FIG, exist_ok=True)

# paleta institucional (relatório de atividades da Passos Mágicos)
AZUL, AZUL_MED, AZUL_CLARO, TERRA = '#001860', '#0060A8', '#78A8C0', '#604830'
SEQ = [AZUL, AZUL_MED, '#1878A8', AZUL_CLARO, TERRA]
PEDRA_COR = {'Quartzo':'#A8C0DC','Ágata':'#7890A8','Ametista':'#0A2472','Topázio':'#001233'}
ORDEM_PEDRA = ['Quartzo','Ágata','Ametista','Topázio']

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi':110, 'savefig.dpi':200, 'savefig.bbox':'tight',
                     'axes.titlesize':13, 'axes.titleweight':'bold', 'axes.titlecolor':AZUL,
                     'axes.labelsize':10, 'font.size':10, 'axes.edgecolor':'#C8D2E0',
                     'grid.color':'#E8EEF6'})

def salvar(fig, nome):
    caminho = os.path.join(PASTA_FIG, f'{nome}.png')
    fig.savefig(caminho, facecolor='white')
    print(f'figura salva: {caminho}')

INDICES = ['IAN','IDA','IEG','IAA','IPS','IPP','IPV']
pd.set_option('display.max_columns', None)

## Base analítica

O recorte é o mesmo do modelo. Reconstruo a partir do `painel_limpo` para deixar o critério explícito e auditável — o total precisa bater com o `painel_modelo`.

In [ ]:
pl = pd.read_csv(PASTA + 'painel_limpo.csv')

MOT_QUALIDADE = ['feature_ausente_sem_imputacao','poucas_features_modelo','ipp_reconstruido_invalido']
b = pl[~pl['excluir_tudo'].astype(bool)].copy()
b = b[~b['motivo_exclusao_modelo'].isin(MOT_QUALIDADE)]
b = b[~b['_movimento_fase_atipico'].astype(bool)]

# qualquer salto de fase (>=2 entre anos consecutivos) sai das análises: as duas pontas do par.
# Decisão conservadora enquanto a mecânica de progressão não é confirmada pela ONG.
_x = pl.dropna(subset=['fase_num'])[['RA','Ano','fase_num']]
_p = _x.merge(_x, on='RA', suffixes=('_T','_T1'))
_p = _p[_p['Ano_T1'] == _p['Ano_T'] + 1]
_s = _p[(_p['fase_num_T1'] - _p['fase_num_T']) >= 2]
pontas = set(zip(_s['RA'], _s['Ano_T'])) | set(zip(_s['RA'], _s['Ano_T1']))
b = b[[k not in pontas for k in zip(b['RA'], b['Ano'])]]

base_traj = b.copy()                              # inclui Fase 8 (apenas trajetória)
base = b[b['fase_num'] != 8].copy()               # análises de índices
base['Genero_pad'] = base['Gênero'].replace({'Menina':'F','Feminino':'F','Menino':'M','Masculino':'M'})

ref = pd.read_csv(PASTA + 'painel_modelo.csv')
print(f'Base analítica: {len(base)} linhas | {base.RA.nunique()} alunos | painel_modelo: {len(ref)}')
assert len(base) == len(ref), 'divergência com o painel_modelo'
print(base.groupby('Ano').size().to_string())

---
## Pergunta 1 — IAN: perfil de defasagem e evolução

O IAN é função-degrau da defasagem (Tabela 41), então leio os dois juntos: o degrau mostra a severidade, a defasagem contínua mostra o movimento dentro de cada faixa.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13,4.4))

dist = (base.groupby(['Ano','Defasagem']).size().unstack(fill_value=0).T)
dist_pct = dist / dist.sum() * 100
dist_pct.plot(kind='bar', ax=ax[0], color=SEQ[:3], width=.78, edgecolor='none')
ax[0].set_title('Distribuição da defasagem por ano')
ax[0].set_xlabel('Defasagem (fase efetiva − fase ideal)'); ax[0].set_ylabel('% dos alunos')
ax[0].legend(title='Ano', frameon=False); ax[0].tick_params(axis='x', rotation=0)

ian_pct = (base.assign(faixa=base['IAN'].map({10.0:'Em fase (10)',5.0:'Moderada (5)',2.5:'Severa (2,5)'}))
           .groupby(['Ano','faixa']).size().unstack(fill_value=0))
ian_pct = (ian_pct.T / ian_pct.sum(axis=1) * 100).T[['Em fase (10)','Moderada (5)','Severa (2,5)']]
ian_pct.plot(kind='bar', stacked=True, ax=ax[1], color=[AZUL_CLARO, AZUL_MED, AZUL], width=.6, edgecolor='none')
ax[1].set_title('Severidade da defasagem (IAN) por ano')
ax[1].set_xlabel('Ano'); ax[1].set_ylabel('% dos alunos')
ax[1].legend(frameon=False, bbox_to_anchor=(1.01,1), loc='upper left'); ax[1].tick_params(axis='x', rotation=0)
for c in ian_pct.columns:
    for i, v in enumerate(ian_pct[c]):
        if v > 6:
            ax[1].text(i, ian_pct.loc[:, :c].sum(axis=1).iloc[i] - v/2, f'{v:.0f}%',
                       ha='center', va='center', color='white', fontsize=9, fontweight='bold')
plt.tight_layout(); salvar(fig, '01_ian_defasagem_por_ano'); plt.show()

print('\n% de alunos com alguma defasagem (D<0) por ano:')
print((base.assign(d=base.Defasagem < 0).groupby('Ano').d.mean()*100).round(1).to_string())
print('\nDefasagem média por ano:')
print(base.groupby('Ano').Defasagem.mean().round(2).to_string())

In [ ]:
# evolução individual: como a defasagem de cada aluno muda de um ano para o outro
tr = base[['RA','Ano','Defasagem']].merge(
        base[['RA','Ano','Defasagem']].assign(Ano=lambda d: d.Ano-1),
        on=['RA','Ano'], suffixes=('_T','_T1'))
tr['mov'] = np.select([tr.Defasagem_T1 < tr.Defasagem_T, tr.Defasagem_T1 > tr.Defasagem_T],
                      ['Agravou','Recuperou'], 'Manteve')
mv = (tr.groupby(['Ano','mov']).size().unstack(fill_value=0))
mv = (mv.T / mv.sum(axis=1) * 100).T

fig, ax = plt.subplots(figsize=(7,4))
mv[['Recuperou','Manteve','Agravou']].plot(kind='barh', stacked=True, ax=ax,
        color=[AZUL_CLARO, '#C8D2E0', AZUL], width=.55, edgecolor='none')
ax.set_title('Movimento da defasagem de um ano para o seguinte')
ax.set_xlabel('% das transições'); ax.set_ylabel('')
ax.set_yticklabels([f'{a}→{a+1}' for a in mv.index]); ax.legend(frameon=False, ncol=3, loc='lower center', bbox_to_anchor=(.5,-.35))
for c in mv.columns:
    for i, v in enumerate(mv[c]):
        if v > 6:
            ax.text(mv.loc[:, :c].sum(axis=1).iloc[i] - v/2, i, f'{v:.0f}%', ha='center', va='center',
                    color='white' if c!='Manteve' else '#12203D', fontsize=9, fontweight='bold')
plt.tight_layout(); salvar(fig, '02_movimento_defasagem'); plt.show()
print(f'\nTransições analisadas: {len(tr)} | agravamento médio: {(tr.mov=="Agravou").mean()*100:.1f}%')

**Insight (P1).** A defasagem é a regra, não a exceção: entre 62% e 69% dos alunos estão atrasados em relação à fase ideal da idade — é o retrato do público que a Associação atende. O indicador melhora ao longo do período (69% em 2022 contra 63% em 2024, com defasagem média passando de −0,92 para −0,83), o que sugere avanço, mas a leitura precisa de cautela: a composição da base muda a cada ano com entradas e saídas, então parte do movimento é renovação de matrícula, não recuperação individual. A leitura pelo IAN mostra que a maioria dos defasados está na faixa moderada (1 a 2 fases). O quadro de transições resolve a ambiguidade olhando o mesmo aluno em dois anos: a maioria mantém a defasagem estável (progride junto com a idade) e cerca de 21% agrava — exatamente a fatia que o modelo da pergunta 9 antecipa.

---
## Pergunta 2 — IDA: desempenho acadêmico por fase e ano

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13,4.4))

sns.boxplot(data=base, x='fase_num', y='IDA', hue='Ano', ax=ax[0], palette=SEQ[:3],
            fliersize=2, linewidth=.9)
ax[0].set_title('IDA por fase e ano'); ax[0].set_xlabel('Fase'); ax[0].set_ylabel('IDA')
ax[0].legend(title='Ano', frameon=False)

med = base.groupby(['Ano','fase_num'])['IDA'].mean().unstack(0)
med.plot(ax=ax[1], marker='o', color=SEQ[:3], linewidth=2)
ax[1].set_title('IDA médio por fase'); ax[1].set_xlabel('Fase'); ax[1].set_ylabel('IDA médio')
ax[1].legend(title='Ano', frameon=False)
plt.tight_layout(); salvar(fig, '03_ida_por_fase_ano'); plt.show()

print('IDA médio por ano:'); print(base.groupby('Ano').IDA.mean().round(2).to_string())
print('\nIDA médio por fase (todos os anos):'); print(base.groupby('fase_num').IDA.agg(['mean','count']).round(2).to_string())

**Insight (P2).** O desempenho não cai de forma contínua ao longo das fases: ele desce da Alfa até a Fase 3 — que é o piso da série, correspondente ao 7º e 8º ano — e volta a subir nas fases seguintes. A Fase 3 concentra a maior dificuldade acadêmica de todo o programa e é o ponto natural para reforço. A recuperação depois dela tem uma explicação a considerar: as fases finais são menores e reúnem os alunos que permaneceram no programa, o que seleciona positivamente o grupo. Entre anos, o IDA médio subiu de 2022 para 2023 e recuou parcialmente em 2024, sem tendência clara de queda.

---
## Pergunta 3 — IEG: engajamento, desempenho (IDA) e ponto de virada (IPV)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15,4.2))

sns.regplot(data=base, x='IEG', y='IDA', ax=ax[0], scatter_kws={'s':9,'alpha':.25,'color':AZUL_MED},
            line_kws={'color':AZUL,'linewidth':2})
ax[0].set_title(f'IEG × IDA  (r = {base.IEG.corr(base.IDA):.2f})')

sns.regplot(data=base, x='IEG', y='IPV', ax=ax[1], scatter_kws={'s':9,'alpha':.25,'color':AZUL_MED},
            line_kws={'color':AZUL,'linewidth':2})
ax[1].set_title(f'IEG × IPV  (r = {base.IEG.corr(base.IPV):.2f})')

q = pd.qcut(base['IEG'], 4, labels=['Q1 (menor)','Q2','Q3','Q4 (maior)'])
comp = base.assign(q=q).groupby('q', observed=True)[['IDA','IPV','INDE_ano']].mean()
comp.plot(kind='bar', ax=ax[2], color=[AZUL, AZUL_MED, AZUL_CLARO], width=.75, edgecolor='none')
ax[2].set_title('Média por quartil de engajamento'); ax[2].set_xlabel('Quartil de IEG'); ax[2].set_ylabel('')
ax[2].legend(frameon=False); ax[2].tick_params(axis='x', rotation=0)
plt.tight_layout(); salvar(fig, '04_ieg_ida_ipv'); plt.show()

print('Correlações com IEG:')
print(base[INDICES + ['INDE_ano']].corr()['IEG'].drop('IEG').round(2).sort_values(ascending=False).to_string())
print('\nMédias por quartil de IEG:'); print(comp.round(2).to_string())

**Insight (P3).** O engajamento conversa fortemente com desempenho e ponto de virada: as correlações do IEG com IDA e com IPV ficam em torno de 0,55, e o efeito é visível em escada — do quartil inferior ao superior de IEG, o IDA médio salta de 5,0 para 7,5 e o IPV de 6,7 para 8,2. O contraste com o IPS é revelador: engajamento e situação psicossocial são praticamente independentes entre si, o que significa que um aluno pode estar engajado e ainda assim vulnerável — e vice-versa. Como o IEG é comportamental e observável no dia a dia, ele é o indicador mais acionável dos três: a equipe percebe a queda de entregas antes de qualquer nota fechar.

---
## Pergunta 4 — IAA: coerência entre autopercepção e desempenho real

In [ ]:
base['gap_auto'] = base['IAA'] - base['IDA']
fig, ax = plt.subplots(1, 3, figsize=(15,4.2))

sns.scatterplot(data=base, x='IDA', y='IAA', ax=ax[0], s=10, alpha=.3, color=AZUL_MED)
lims = [0, 10.2]; ax[0].plot(lims, lims, '--', color=TERRA, linewidth=1.4, label='IAA = IDA')
ax[0].set_xlim(lims); ax[0].set_ylim(lims); ax[0].legend(frameon=False)
ax[0].set_title(f'Autoavaliação × desempenho  (r = {base.IAA.corr(base.IDA):.2f})')

sns.scatterplot(data=base, x='IEG', y='IAA', ax=ax[1], s=10, alpha=.3, color=AZUL_MED)
ax[1].plot(lims, lims, '--', color=TERRA, linewidth=1.4)
ax[1].set_xlim(lims); ax[1].set_ylim(lims)
ax[1].set_title(f'Autoavaliação × engajamento  (r = {base.IAA.corr(base.IEG):.2f})')

g = base.groupby('fase_num')[['IAA','IDA','IEG']].mean()
g.plot(ax=ax[2], marker='o', color=[AZUL, AZUL_MED, AZUL_CLARO], linewidth=2)
ax[2].set_title('IAA, IDA e IEG médios por fase'); ax[2].set_xlabel('Fase'); ax[2].set_ylabel('')
ax[2].legend(frameon=False)
plt.tight_layout(); salvar(fig, '05_iaa_coerencia'); plt.show()

print(f'IAA médio: {base.IAA.mean():.2f} | IDA médio: {base.IDA.mean():.2f} | IEG médio: {base.IEG.mean():.2f}')
print(f'Acima do próprio desempenho (IAA > IDA): {(base.gap_auto>0).mean()*100:.1f}% dos alunos')
print()
print('Gap IAA−IDA médio por fase:'); print(base.groupby('fase_num').gap_auto.mean().round(2).to_string())

**Insight (P4).** A autopercepção é descolada das duas réguas objetivas que o enunciado pede: a correlação do IAA é fraca tanto com o desempenho (r = 0,14) quanto com o engajamento (r = 0,16) — e sistematicamente mais alta que ambos (média 8,7 contra 6,4 de IDA), com quase 9 em cada 10 alunos se avaliando acima da própria nota. Isso não é distorção: o IAA registra como o aluno se sente consigo, com a família, com os amigos e com a Associação, dimensões que não passam pela prova nem pela lista de entregas. O gap é maior justamente na Fase 3, a de menor desempenho — a autopercepção sustenta o aluno quando a nota não sustenta. O dado acionável está nos extremos: IAA alto com IDA baixo aponta quem pode não estar percebendo a própria dificuldade; IAA baixo apesar de bom desempenho e engajamento sinaliza autoestima frágil — perfis que pedem conversas opostas.

---
## Pergunta 5 — IPS: padrões psicossociais que antecedem quedas de desempenho

Leitura prospectiva, como a pergunta exige: os índices **do ano T** contra os dois desfechos do ano seguinte — queda de desempenho (IDA) e queda de engajamento (IEG).

In [ ]:
par = base[['RA','Ano','IPS','IAA','IEG','IPP','IPV','IDA','Defasagem']].merge(
        base[['RA','Ano','IDA','IEG','Defasagem']].assign(Ano=lambda d: d.Ano-1),
        on=['RA','Ano'], suffixes=('_T','_T1'))
par['caiu_ida'] = par['IDA_T1'] < par['IDA_T']
par['caiu_ieg'] = par['IEG_T1'] < par['IEG_T']
par['agravou'] = par['Defasagem_T1'] < par['Defasagem_T']

# cada índice do ano T contra os DOIS desfechos do ano seguinte
prev = ['IPS','IAA','IEG_T','IPP','IPV','IDA_T']
corr2 = pd.DataFrame({
    'queda de IDA em T+1': par[prev].corrwith(par.caiu_ida.astype(int)),
    'queda de IEG em T+1': par[prev].corrwith(par.caiu_ieg.astype(int))}).round(3)

fig, ax = plt.subplots(1, 2, figsize=(13,4.2))
corr2.plot(kind='barh', ax=ax[0], color=[AZUL_CLARO, AZUL], width=.75, edgecolor='none')
ax[0].axvline(0, color='#C8D2E0', linewidth=1)
ax[0].set_title('Cada índice no ano T × queda no ano seguinte'); ax[0].set_xlabel('correlação')
ax[0].legend(frameon=False)

qs = pd.qcut(par['IPS'], 4, labels=['Q1 (menor)','Q2','Q3','Q4 (maior)'])
taxa = par.assign(q=qs).groupby('q', observed=True)[['caiu_ida','caiu_ieg','agravou']].mean()*100
taxa.plot(kind='bar', ax=ax[1], color=[AZUL_CLARO, AZUL_MED, AZUL], width=.78, edgecolor='none')
ax[1].set_title('Desfecho no ano seguinte por quartil de IPS')
ax[1].set_xlabel('Quartil de IPS (ano T)'); ax[1].set_ylabel('% dos alunos')
ax[1].legend(['Queda de IDA','Queda de IEG','Agravamento'], frameon=False); ax[1].tick_params(axis='x', rotation=0)
plt.tight_layout(); salvar(fig, '06_ips_antecede_queda'); plt.show()

print(f'Transições: {len(par)} | queda de IDA em T+1: {par.caiu_ida.mean()*100:.1f}% | queda de IEG: {par.caiu_ieg.mean()*100:.1f}%')
print()
print('Correlações (índice no ano T × desfecho em T+1):'); print(corr2.to_string())
print()
print('IPS no ano T — manteve IEG:', round(par[~par.caiu_ieg].IPS.mean(),2),
      '| caiu de IEG:', round(par[par.caiu_ieg].IPS.mean(),2))

**Insight (P5).** O enunciado pergunta por quedas de desempenho **e** de engajamento, e as respostas são diferentes. Para a **nota**, o IPS não antecede nada: a diferença entre quem cai e quem não cai é de 0,13 ponto, os quartis não formam gradiente, e o único preditor relevante é o próprio IDA anterior (quem caiu tinha IDA médio 7,3 contra 6,2 — reversão à média: quem está no alto tende a cair). Para o **engajamento**, aparece o sinal que a pergunta procura: alunos com IPS mais baixo hoje desengajam um pouco mais amanhã (IPS 5,74 entre quem caiu de IEG contra 6,20 entre quem manteve; r = −0,12) — fraco, mas na direção esperada e consistente com o papel do índice. A leitura operacional: o canal psicossocial se manifesta primeiro no comportamento (entregas, presença), não na prova; e como nenhum índice isolado prevê bem, a combinação de todos — que é o que o modelo da pergunta 9 faz — segue sendo a forma correta de ler risco.

---
## Pergunta 6 — IPP: confirma ou contradiz a defasagem apontada pelo IAN?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13,4.2))

ordem_ian = [2.5, 5.0, 10.0]
rot_ian = {2.5:'Severa (2,5)', 5.0:'Moderada (5)', 10.0:'Em fase (10)'}
d = base.assign(faixa=base['IAN'].map(rot_ian))
sns.boxplot(data=d, x='faixa', y='IPP', ax=ax[0], order=[rot_ian[i] for i in ordem_ian],
            hue='faixa', legend=False, palette=[AZUL, AZUL_MED, AZUL_CLARO], fliersize=2, linewidth=.9)
ax[0].set_title('IPP por faixa de defasagem (IAN)'); ax[0].set_xlabel(''); ax[0].set_ylabel('IPP')

corr = base.groupby('Ano').apply(lambda g: g['IPP'].corr(g['Defasagem']), include_groups=False)
tab = base.groupby('faixa' if 'faixa' in base else base['IAN'].map(rot_ian))[['IPP','IDA','IEG']].mean()
tab = tab.reindex([rot_ian[i] for i in ordem_ian])
tab.plot(kind='bar', ax=ax[1], color=[AZUL, AZUL_MED, AZUL_CLARO], width=.75, edgecolor='none')
ax[1].set_title('Médias por faixa de defasagem'); ax[1].set_xlabel(''); ax[1].set_ylabel('')
ax[1].legend(frameon=False); ax[1].tick_params(axis='x', rotation=0)
plt.tight_layout(); salvar(fig, '07_ipp_vs_ian'); plt.show()

print('IPP médio por faixa de IAN:'); print(tab['IPP'].round(2).to_string())
print(f'\nCorrelação IPP × Defasagem por ano:'); print(corr.round(2).to_string())

**Insight (P6).** As duas leituras são parcialmente independentes: alunos em defasagem severa têm IPP mais baixo em média, mas a diferença entre as faixas é modesta e há alunos com defasagem severa e IPP alto. A interpretação é direta — a defasagem mede o *atraso acumulado* (histórico), enquanto o IPP mede o *desenvolvimento observado hoje* pelos educadores. Um aluno pode estar dois anos atrasado e evoluindo bem; é esse aluno que o programa está recuperando, e olhar só para o IAN o classificaria como caso grave.

---
## Pergunta 7 — IPV: comportamentos que mais influenciam o ponto de virada

In [ ]:
par_ipv = base[['RA','Ano','IPV','IEG','IDA','IPP','IPS','IAA']].merge(
        base[['RA','Ano','IPV']].assign(Ano=lambda d: d.Ano-1),
        on=['RA','Ano'], suffixes=('_T','_T1'))

idx5 = ['IEG','IDA','IPP','IPS','IAA']
cmp_t = pd.DataFrame({
    'IPV no mesmo ano': base[idx5 + ['IPV']].corr()['IPV'].drop('IPV'),
    'IPV no ano seguinte': par_ipv[idx5].corrwith(par_ipv['IPV_T1'])}).round(2)

fig, ax = plt.subplots(1, 2, figsize=(13,4.2))
cmp_t.sort_values('IPV no mesmo ano').plot(kind='barh', ax=ax[0],
        color=[AZUL_CLARO, AZUL], width=.75, edgecolor='none')
ax[0].set_title('Associação de cada indicador com o IPV — hoje e no ano seguinte')
ax[0].set_xlabel('correlação'); ax[0].legend(frameon=False)

qv = base.assign(q=pd.qcut(base['IPV'], 4, labels=['Q1 (menor)','Q2','Q3','Q4 (maior)']))
perfil = qv.groupby('q', observed=True)[['IEG','IDA','IAA','IPS','IPP']].mean()
sns.heatmap(perfil.T, annot=True, fmt='.1f', cmap='Blues', ax=ax[1], cbar=False, linewidths=.5, linecolor='white')
ax[1].set_title('Perfil médio por quartil de IPV'); ax[1].set_xlabel('Quartil de IPV'); ax[1].set_ylabel('')
plt.tight_layout(); salvar(fig, '08_ipv_influencias'); plt.show()

par_ipv['dIPV'] = par_ipv['IPV_T1'] - par_ipv['IPV_T']
print(f'Pares ano a ano: {len(par_ipv)}')
print()
print('Correlação com o GANHO de IPV (T → T+1):')
print(par_ipv[idx5 + ['IPV_T']].corrwith(par_ipv['dIPV']).round(2).sort_values(ascending=False).to_string())
print()
print('Perfil por quartil de IPV:'); print(perfil.round(2).to_string())

**Insight (P7).** No retrato do mesmo ano, três indicadores empatam na associação com o ponto de virada (IDA, IEG e IPP, todos entre 0,52 e 0,55), com autoavaliação e IPS de fora — o IPV reflete o que os educadores observam em sala, não o que o aluno sente. A dimensão que o enunciado pede, **ao longo do tempo**, acrescenta duas camadas: o desempenho de hoje é o que mais se associa ao IPV do ano seguinte (r = 0,37, acima até do próprio IPV atual, 0,32) — a virada de amanhã se constrói sobre a base acadêmica de hoje; e, olhando o **ganho** de IPV entre anos, o IPS é o único índice com associação positiva relevante (+0,25) — suporte psicossocial hoje acompanha evolução do ponto de virada amanhã (leitura possível porque o IPS é quase independente do IPV corrente, o que afasta o artefato de régua; já a forte correlação negativa do próprio IPV com o ganho, −0,53, é regressão à média e teto de escala, não piora). Em conjunto: constância e desempenho constroem o nível do IPV; o psicossocial aparece no movimento.

---
## Pergunta 8 — Multidimensionalidade: quais combinações elevam o INDE

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13,4.6))

cm = base[INDICES + ['INDE_ano']].corr()
mask = np.triu(np.ones_like(cm, dtype=bool), k=1)
sns.heatmap(cm, mask=mask, annot=True, fmt='.2f', cmap='Blues', center=0, ax=ax[0],
            cbar=False, linewidths=.5, linecolor='white', annot_kws={'size':8})
ax[0].set_title('Correlação entre os indicadores e o INDE')

# contribuição efetiva: peso documental x dispersão observada
PESOS = {'IAN':.1,'IDA':.2,'IEG':.2,'IAA':.1,'IPS':.1,'IPP':.1,'IPV':.2}
contrib = pd.DataFrame({
    'peso na fórmula': pd.Series(PESOS),
    'variação explicada': pd.Series({c: PESOS[c]*base[c].std() for c in INDICES})
})
contrib['variação explicada'] /= contrib['variação explicada'].sum()
contrib['peso na fórmula'] /= contrib['peso na fórmula'].sum()
contrib.sort_values('variação explicada').plot(kind='barh', ax=ax[1], color=[AZUL_CLARO, AZUL], edgecolor='none')
ax[1].set_title('Peso na fórmula × influência real na variação do INDE')
ax[1].xaxis.set_major_formatter(mticker.PercentFormatter(1.0)); ax[1].set_xlabel(''); ax[1].legend(frameon=False)
plt.tight_layout(); salvar(fig, '09_multidimensionalidade_inde'); plt.show()

alto = base['INDE_ano'] >= base['INDE_ano'].quantile(.75)
comp = pd.DataFrame({'INDE alto (25% sup.)': base[alto][INDICES].mean(),
                     'demais': base[~alto][INDICES].mean()})
comp['diferença'] = comp.iloc[:,0] - comp.iloc[:,1]
print('Perfil dos alunos com INDE no quartil superior:')
print(comp.round(2).sort_values('diferença', ascending=False).to_string())

In [ ]:
# combinações literais de IDA + IEG + IPS + IPP (acima/abaixo da mediana de cada um)
alto = {c: base[c] >= base[c].median() for c in ['IDA','IEG','IPS','IPP']}
cmb = base.assign(**{c + '_alto': alto[c] for c in alto})
g = (cmb.groupby(['IDA_alto','IEG_alto','IPS_alto','IPP_alto'])['INDE_ano'].agg(['mean','count'])
        .sort_values('mean', ascending=False))
g.index = [' + '.join(n for n, f in zip(['IDA','IEG','IPS','IPP'], flags) if f) or 'nenhum alto'
           for flags in g.index]

fig, ax = plt.subplots(figsize=(9,5.4))
cores = [AZUL if i < 3 else (TERRA if i >= len(g)-3 else AZUL_CLARO) for i in range(len(g))]
g['mean'].iloc[::-1].plot(kind='barh', ax=ax, color=cores[::-1], edgecolor='none')
for i, (v, n) in enumerate(zip(g['mean'].iloc[::-1], g['count'].iloc[::-1])):
    ax.text(v + .04, i, f'{v:.2f}  (n={n})', va='center', fontsize=8.5)
ax.set_title('INDE médio por combinação de indicadores acima da mediana')
ax.set_xlabel('INDE médio'); ax.set_xlim(5.6, 9.3)
plt.tight_layout(); salvar(fig, '09b_combinacoes_inde'); plt.show()

so_ida = cmb[cmb['IDA_alto'] & ~cmb['IEG_alto']]['INDE_ano'].mean()
so_ieg = cmb[~cmb['IDA_alto'] & cmb['IEG_alto']]['INDE_ano'].mean()
ambos  = cmb[cmb['IDA_alto'] & cmb['IEG_alto']]['INDE_ano'].mean()
nenhum = cmb[~cmb['IDA_alto'] & ~cmb['IEG_alto']]['INDE_ano'].mean()
print(f'Dupla central IDA × IEG — nenhum alto: {nenhum:.2f} | só IDA: {so_ida:.2f} | só IEG: {so_ieg:.2f} | ambos: {ambos:.2f}')

**Insight (P8).** As combinações respondem à pergunta com clareza de régua: alunos acima da mediana **nos quatro indicadores ao mesmo tempo** (IDA, IEG, IPS e IPP) têm INDE médio de 8,41, contra 6,11 de quem está abaixo em todos — 2,3 pontos de distância, quase a largura de duas pedras. A hierarquia das 16 combinações mostra o núcleo: as melhores têm sempre IDA e IEG juntos, e a dupla é praticamente aditiva (nenhum alto 6,41 → só IDA 7,48 → só IEG 7,34 → ambos 8,15) — os ganhos se somam, sem exigir condição especial. O IPS é o que menos move a régua sozinho (+0,3), coerente com sua independência dos demais; e o IAA, apesar de pesar 10% na fórmula, quase não diferencia alunos por estar concentrado no topo. A comparação entre peso nominal e influência real fecha a leitura: indicadores com dispersão (IDA, IAN, IEG) governam o INDE; elevar o índice passa por atuar em conjunto sobre desempenho e engajamento.

---
## Pergunta 10 — Efetividade: melhora consistente ao longo do ciclo de pedras

Três recortes: distribuição das pedras por ano, trajetória individual de quem tem 2+ anos de registro, e o desfecho dos alunos que chegaram à universidade. As pedras aqui são as **rederivadas pelos cortes exatos do documento** (6,110 / 7,154 / 8,198); os rótulos originais da Associação ficam preservados em `Pedra_ano_base` e divergem em parte das linhas.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13,4.4))

pd_ano = base.groupby(['Ano','Pedra_ano']).size().unstack(fill_value=0)
pd_ano = pd_ano.reindex(columns=[c for c in ORDEM_PEDRA if c in pd_ano.columns])
(pd_ano.T / pd_ano.sum(axis=1) * 100).T.plot(kind='bar', stacked=True, ax=ax[0],
        color=[PEDRA_COR[c] for c in pd_ano.columns], width=.6, edgecolor='none')
ax[0].set_title('Distribuição das pedras por ano'); ax[0].set_xlabel('Ano'); ax[0].set_ylabel('% dos alunos')
ax[0].legend(frameon=False, bbox_to_anchor=(1.01,1), loc='upper left'); ax[0].tick_params(axis='x', rotation=0)

ev = base.groupby('Ano')['INDE_ano'].agg(['mean','median'])
ev.plot(ax=ax[1], marker='o', color=[AZUL, AZUL_MED], linewidth=2)
ax[1].set_title('INDE médio e mediano por ano'); ax[1].set_xlabel('Ano'); ax[1].set_ylabel('INDE')
ax[1].set_xticks(sorted(base.Ano.unique())); ax[1].legend(['média','mediana'], frameon=False)
plt.tight_layout(); salvar(fig, '10_pedras_e_inde_por_ano'); plt.show()

print('INDE por ano:'); print(ev.round(2).to_string())

In [ ]:
# variação do INDE e movimento de pedra — com os limites estruturais da escala à vista
tj = base[['RA','Ano','INDE_ano','Pedra_ano']].merge(
        base[['RA','Ano','INDE_ano','Pedra_ano']].assign(Ano=lambda d: d.Ano-1),
        on=['RA','Ano'], suffixes=('_T','_T1'))
tj['delta'] = tj['INDE_ano_T1'] - tj['INDE_ano_T']
ordem = {p:i for i,p in enumerate(ORDEM_PEDRA)}
tj['mov'] = np.select([tj.Pedra_ano_T1.map(ordem) > tj.Pedra_ano_T.map(ordem),
                       tj.Pedra_ano_T1.map(ordem) < tj.Pedra_ano_T.map(ordem)], ['Subiu','Desceu'], 'Manteve')

fig, ax = plt.subplots(1, 2, figsize=(13,4.4))
sns.boxplot(data=tj, x='Pedra_ano_T', y='delta', order=ORDEM_PEDRA, hue='Pedra_ano_T',
            hue_order=ORDEM_PEDRA, legend=False, palette=PEDRA_COR, ax=ax[0],
            fliersize=2, linewidth=.9)
ax[0].axhline(0, color=TERRA, linestyle='--', linewidth=1.2)
ax[0].set_title('Variação do INDE por pedra de partida')
ax[0].set_xlabel('Pedra no ano T'); ax[0].set_ylabel('Δ INDE (T → T+1)')

ras3 = base.groupby('RA')['Ano'].nunique()
ras3 = ras3[ras3 == 3].index
c3 = base[base.RA.isin(ras3)]
c3.groupby('Ano')['INDE_ano'].mean().plot(ax=ax[1], marker='o', color=AZUL, linewidth=2.2)
ax[1].set_title(f'Coorte fechada — INDE médio dos mesmos {len(ras3)} alunos')
ax[1].set_xlabel('Ano'); ax[1].set_ylabel('INDE médio'); ax[1].set_xticks([2022,2023,2024])
plt.tight_layout(); salvar(fig, '11_trajetoria_inde_pedra'); plt.show()

mp = tj.groupby('Pedra_ano_T')['mov'].value_counts(normalize=True).unstack(fill_value=0)*100
mp = mp.reindex(ORDEM_PEDRA).reindex(columns=['Subiu','Manteve','Desceu'])
print('Movimento de pedra por ponto de partida (%):')
print('(limites estruturais da escala: de Quartzo não há como descer; de Topázio não há como subir)')
print(mp.round(1).to_string())
print()
print('Saldo nas pedras intermediárias, onde os dois movimentos são possíveis:')
for p in ['Ágata','Ametista']:
    sub = tj[tj.Pedra_ano_T == p]
    print(f'  {p}: sobe {(sub.mov=="Subiu").mean()*100:.1f}% | desce {(sub.mov=="Desceu").mean()*100:.1f}% | n={len(sub)}')
print()
w = c3.pivot_table(index='RA', columns='Ano', values='INDE_ano')
print(f'Coorte fechada ({len(ras3)} alunos): INDE 2022 {w[2022].mean():.2f} → 2023 {w[2023].mean():.2f} → 2024 {w[2024].mean():.2f}')
print(f'  terminam acima de onde começaram: {(w[2024] > w[2022]).mean()*100:.1f}% | Δ 2022→2024 mediano: {(w[2024]-w[2022]).median():+.2f}')

In [ ]:
# universitários — bloco de trajetória, não de índices.
# Filtro próprio: fabricados e movimentos atípicos de fase ficam de fora; os critérios de
# features do modelo não se aplicam aqui (contar percurso pede fase e defasagem, não IAA/IPP).
bt = pl[~pl['excluir_tudo'].astype(bool)]
bt = bt[[k not in pontas for k in zip(bt['RA'], bt['Ano'])]]

val = []
for ra in bt[bt['fase_num']==8]['RA'].unique():
    sub = bt[bt['RA']==ra].sort_values('Ano')
    ant, f8r = sub[sub['fase_num']<8], sub[sub['fase_num']==8]
    if len(ant)==0 or len(f8r)==0: continue
    ult, prim8 = ant.iloc[-1], f8r.iloc[0]
    if (prim8['Ano'] - ult['Ano'] == 1) and (ult['fase_num'] == 7):
        val.append({'RA':ra,'ano_ingresso_f8':int(prim8['Ano']),'defasagem_antes':ult['Defasagem'],
                    'INDE_antes':ult['INDE_ano'],'Pedra_antes':ult['Pedra_ano'],'IDA_antes':ult['IDA']})
uni = pd.DataFrame(val)

fig, ax = plt.subplots(1, 2, figsize=(12,3.8))
(uni['defasagem_antes'].value_counts().sort_index()
 .plot(kind='bar', ax=ax[0], color=AZUL, width=.6, edgecolor='none'))
ax[0].set_title('Defasagem no último ano antes da universidade')
ax[0].set_xlabel('Defasagem'); ax[0].set_ylabel('alunos'); ax[0].tick_params(axis='x', rotation=0)

pc = uni['Pedra_antes'].value_counts().reindex(ORDEM_PEDRA).dropna()
pc.plot(kind='bar', ax=ax[1], color=[PEDRA_COR[i] for i in pc.index], width=.6, edgecolor='none')
ax[1].set_title('Pedra no último ano antes da universidade')
ax[1].set_xlabel(''); ax[1].set_ylabel('alunos'); ax[1].tick_params(axis='x', rotation=0)
plt.tight_layout(); salvar(fig, '12_universitarios_trajetoria'); plt.show()

print(f'Universitários com trajetória documentada (Fase 7 → 8 em anos consecutivos): {len(uni)}')
print(f'  vinham defasados: {int((uni.defasagem_antes<0).sum())} | em fase: {int((uni.defasagem_antes==0).sum())}')
print(f'  com INDE registrado no último ano: {int(uni.INDE_antes.notna().sum())} | INDE médio: {uni.INDE_antes.mean():.2f}')
print()
print('Notas: (1) os indicadores da Fase 8 não são coletados pela Associação (apenas o IAN) —')
print('a análise é de trajetória, não de desempenho corrente; (2) um caso adicional, com')
print('trajetória 5→7→8, fica de fora pela regra de movimentos atípicos de fase.')

**Insight (P10).** Respondendo à pergunta como formulada — há melhora consistente ao longo do ciclo de pedras? — a resposta honesta é: **em média, não; em subgrupos e desfechos, sim.** A evidência mais limpa é a coorte fechada: acompanhando os mesmos 328 alunos por três anos, o INDE fica estável (7,47 → 7,57 → 7,41), com 54% terminando acima de onde começaram e mediana de variação próxima de zero. O quadro de movimento de pedra exige leitura com os limites estruturais da escala à vista: de Quartzo é impossível descer e de Topázio é impossível subir, então esses extremos não informam efetividade por si. Nas pedras intermediárias, onde os dois movimentos existem, o saldo é positivo em Ágata (33% sobem contra 20% que descem) e neutro em Ametista — e o Δ INDE decrescente conforme a pedra de partida sobe é o padrão esperado de regressão à média somado ao teto da régua. Onde a melhora **é** identificável: na base da distribuição (Ágata; e a taxa de subida de 66% em Quartzo é alta mesmo sabendo que descer não era opção) e nos desfechos de trajetória — a queda da proporção de defasados ao longo dos anos e os universitários que chegaram à Fase 8 vindos de defasagem. Sem um grupo de comparação externo, estabilidade num público de alta vulnerabilidade não demonstra ineficácia — mas afirmar melhora generalizada também não seria sustentado pelos dados. A régua de pedras mede posição numa escala truncada; para efetividade, os desfechos de percurso são a evidência mais sólida desta base.

---
## Pergunta 11 — Insights adicionais e sugestões para a Associação

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13,4.2))

base['tempo_prog'] = base['Ano'] - pd.to_numeric(base['Ano ingresso'], errors='coerce')
tp = base[base['tempo_prog'].between(0, 8)]
tp.groupby('tempo_prog')[['INDE_ano','IDA','IEG']].mean().plot(ax=ax[0], marker='o',
        color=[AZUL, AZUL_MED, AZUL_CLARO], linewidth=2)
ax[0].set_title('Indicadores por tempo de programa'); ax[0].set_xlabel('anos desde o ingresso'); ax[0].set_ylabel('')
ax[0].legend(frameon=False)

rede = base.copy()
rede['rede'] = np.where(rede['Instituicao'].astype(str).str.contains('blica', case=False, na=False),
                        'Pública', 'Outras/Particular')
rd = rede.groupby('rede')[['INDE_ano','IDA','IEG','IPS']].mean()
rd.plot(kind='bar', ax=ax[1], color=SEQ[:4], width=.75, edgecolor='none')
ax[1].set_title('Indicadores por rede de ensino'); ax[1].set_xlabel(''); ax[1].set_ylabel('')
ax[1].legend(frameon=False); ax[1].tick_params(axis='x', rotation=0)
plt.tight_layout(); salvar(fig, '13_tempo_programa_e_rede'); plt.show()

print('Média por tempo de programa:')
print(tp.groupby('tempo_prog')[['INDE_ano','IDA','IEG']].agg(['mean','count']).round(2).to_string())
print('\nPor rede:'); print(rd.round(2).to_string())

In [ ]:
# achados de qualidade de registro (subproduto do tratamento, acionáveis pela ONG)
pl_full = pd.read_csv(PASTA + 'painel_limpo.csv')
_fib = pl_full['Fase_Ideal_base'].astype(str)
_ideal_base_num = pd.to_numeric(_fib.str.extract('([0-9]+)')[0], errors='coerce')
_ideal_base_num = _ideal_base_num.mask(_fib.str.upper().str.contains('ALFA'), 0)  # rótulos ALFA trazem o ano escolar entre parênteses
achados = pd.DataFrame([
    {'achado':'Fase ideal divergente da Tabela 4 na base original',
     'linhas': int((_ideal_base_num.fillna(-1) != pl_full['fase_ideal_num'].fillna(-1)).sum())},
    {'achado':'Defasagem alterada após recálculo',
     'linhas': int((pl_full['Defasagem'] != pl_full['Defasagem_base']).sum())},
    {'achado':'IAA abaixo do piso da escala (Tabela 40)',
     'linhas': int(pl_full['_iaa_invalido'].sum())},
    {'achado':'INDE contaminado por IAA inválido (anulado)',
     'linhas': int(pl_full['_inde_contaminado'].sum())},
    {'achado':'Fase corrigida (registro além da fase ideal)',
     'linhas': int((pl_full['fase_num'] != pl_full['fase_num_base']).sum())},
    {'achado':'Indicadores não coletados na Fase 8 (apenas IAN)',
     'linhas': int((pl_full['fase_num']==8).sum())},
])
print(achados.to_string(index=False))

**Insight (P11).** Duas leituras dos painéis acima, com a cautela devida: o INDE levemente declinante com o tempo de programa não deve ser lido como desgaste — tempo de casa anda junto com fase e idade, e as fases intermediárias são as de menor desempenho (efeito composição, não causal). Já o recorte por rede mostra diferença real de partida: alunos de escola pública — a grande maioria — têm IDA quase 0,7 ponto abaixo dos demais, com IPS mais alto; o reforço acadêmico tem endereço. Abaixo, os achados de qualidade de registro e as sugestões.

**Sugestões para a Associação.**

**1. Priorização por risco, não por média.** O modelo da pergunta 9 classifica cada aluno em Baixo/Médio/Alto risco de agravar a defasagem. Concentrar o acompanhamento no grupo de risco Alto alcança metade dos casos reais de agravamento em uma fração dos alunos — é o uso mais direto do trabalho.

**2. Usar o IEG e o IPS como alarme antecipado.** Ambos caem antes da nota (pergunta 5) e são observáveis pela equipe no dia a dia. Uma rotina simples de sinalização quando qualquer um deles fica no quartil inferior antecipa a intervenção em um ciclo inteiro.

**3. Coletar indicadores dos universitários.** Hoje a Fase 8 registra apenas o IAN — os demais índices ficam vazios. É justamente a etapa que comprova o sucesso do programa, e ela é a menos medida. Um formulário reduzido (engajamento, vínculo com a Associação, desempenho no curso) tornaria o resultado final demonstrável.

**4. Padronizar o cadastro na origem.** Os achados de qualidade acima não são erros de análise: são divergências entre o registro e as próprias regras da Associação (fase ideal, escala da autoavaliação, formato da fase). Validações simples no momento do preenchimento eliminam a maior parte deles.

**5. Ler autopercepção e desempenho juntos.** O IAA alto com IDA baixo (pergunta 4) identifica alunos que não estão percebendo a própria dificuldade — um perfil que passa despercebido quando cada indicador é olhado isoladamente.

---
### Encerramento

Figuras salvas em `reports/figuras/` — prontas para o storytelling. As perguntas 1–8, 10 e 11 são respondidas aqui; a 9 é respondida pelo modelo do notebook 02.

**Recorte usado:** 2.566 linhas, idêntico ao do modelo. Ficam de fora matrículas em processamento, linhas sem features completas, movimentos atípicos de fase (decisão conservadora, a revisar após confirmação da mecânica de progressão com a Associação) e a Fase 8 nas análises de índices — esta última por ausência estrutural de coleta, o que é em si um dos achados da pergunta 11.